# Library

In [1]:
import sys, warnings
sys.path.append('..')
from src.preprocessing import full_preprocessing_pipeline
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from src.preprocessing import full_preprocessing_pipeline
from src.features import (
    add_engineered_features, 
    encode_scenario, 
    prepare_scenarios, 
    split_and_scale,
    prepare_all
)

# Load Data

In [2]:
df_all = pd.read_csv('D:\Dibimbing.Id\Supply Chain\dataset\DataCoSupplyChainDataset.csv', encoding='latin1')

# membaca data yang sudah melewati preprocessing
df = full_preprocessing_pipeline('D:\Dibimbing.Id\Supply Chain\dataset\DataCoSupplyChainDataset.csv')

# menambahkan fitur baru
df = add_engineered_features(df)  

Dataset berjumlah 180519 baris dan 53 kolom
Kolom yang telah dihapus (22): ['Delivery Status', 'shipping date (DateOrders)', 'Customer Email', 'Customer Fname', 'Customer Lname', 'Customer Password', 'Customer Street', 'Customer Id', 'Order Customer Id', 'Order Id', 'Order Item Id', 'Order Item Cardprod Id', 'Product Card Id', 'Product Category Id', 'Category Id', 'Department Id', 'Product Image', 'Customer Zipcode', 'Latitude', 'Longitude', 'Product Description', 'Order Zipcode']
Tidak ada missing values.

Kolom yang memiliki missing value lebih dari 50%: []
Jumlah kolom sebelum didrop: 31
Jumlah kolom setelah didrop: 31
Tidak ada data yang duplicates.
Fitur redundan yang didrop: ['Order Profit Per Order', 'Order Item Product Price', 'Order Item Total']
Kolom kategorik yang didrop (10): ['Category Name', 'Customer Country', 'Customer Segment', 'Department Name', 'Market', 'Order City', 'Order State', 'order date (DateOrders)', 'Order Status', 'Order Country']
Kolom konstan di-drop: ['

In [3]:
# Target yang digunakan
target = 'Late_delivery_risk'

# Fitur pada Skenario 1 : Pre-Shipment (info minimal saat order masuk)
features_1 = [
    'Days for shipment (scheduled)',
    'Shipping Mode',
    'Type',
    'Order Item Quantity',
    'Product Price',
    'Order Item Discount Rate',
]

features_2 = [
    'Days for shipment (scheduled)',
    'Shipping Mode',
    'Type',
    'Order Item Quantity',
    'Order Item Discount Rate',
    'Product Price',
    'Order Item Profit Ratio',
    'discount_per_unit',
    'Order Region',
    'Customer State',
]

shipping_order = {
    'Same Day'      : 0,
    'First Class'   : 1,
    'Second Class'  : 2,
    'Standard Class': 3,
}

# Encode

In [4]:
print(df.columns.tolist())

['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Late_delivery_risk', 'Customer City', 'Customer State', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Region', 'Product Name', 'Product Price', 'Shipping Mode', 'discount_per_unit']


In [5]:
# Hasil Encode 2 Skenario
df_s1, df_s2 = prepare_scenarios(df)

Fitur baru ditambahkan: discount_per_unit
OHE cols: ['Type']
Dtypes sebelum OHE: Type    str
dtype: object
OHE cols: ['Type', 'Order Region', 'Customer State']
Dtypes sebelum OHE: Type              str
Order Region      str
Customer State    str
dtype: object
Skenario 1 setelah encoding: (180519, 9)
Skenario 2 setelah encoding: (180519, 78)


In [6]:
# perbandingan scaling pada skenario 1
df_enc = encode_scenario(df, features_1)  
X = df_enc.drop(columns=target)
y = df_enc[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = X_tr.select_dtypes(include='number').columns.tolist()
results_sc = {}
for method in ['Tanpa Scaling', 'StandardScaler', 'MinMaxScaler']:
    Xtr, Xte = X_tr.copy(), X_te.copy()
    if method == 'StandardScaler':
        sc = StandardScaler()
        Xtr[num_cols] = sc.fit_transform(Xtr[num_cols])
        Xte[num_cols] = sc.transform(Xte[num_cols])
    elif method == 'MinMaxScaler':
        sc = MinMaxScaler()
        Xtr[num_cols] = sc.fit_transform(Xtr[num_cols])
        Xte[num_cols] = sc.transform(Xte[num_cols])
    lr   = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(Xtr, y_tr)
    pred = lr.predict(Xte)
    results_sc[method] = {'Accuracy': round(accuracy_score(y_te, pred),4),
                          'F1-Score': round(f1_score(y_te, pred),4)}

print(f"{'Method':<18} {'Accuracy':>10} {'F1-Score':>10}")
print('-'*42)
for m, v in results_sc.items():
    note = ' ← DIPILIH' if m=='Tanpa Scaling' else ''
    print(f"{m:<18} {v['Accuracy']:>10} {v['F1-Score']:>10}{note}")


OHE cols: ['Type']
Dtypes sebelum OHE: Type    str
dtype: object
Method               Accuracy   F1-Score
------------------------------------------
Tanpa Scaling          0.6919     0.6767 ← DIPILIH
StandardScaler         0.6919     0.6767
MinMaxScaler           0.6919     0.6767


In [7]:
# perbandingan scaling pada skenario 2
df_enc = encode_scenario(df, features_2) 
X = df_enc.drop(columns=target)
y = df_enc[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = X_tr.select_dtypes(include='number').columns.tolist()
results_sc = {}
for method in ['Tanpa Scaling', 'StandardScaler', 'MinMaxScaler']:
    Xtr, Xte = X_tr.copy(), X_te.copy()
    if method == 'StandardScaler':
        sc = StandardScaler()
        Xtr[num_cols] = sc.fit_transform(Xtr[num_cols])
        Xte[num_cols] = sc.transform(Xte[num_cols])
    elif method == 'MinMaxScaler':
        sc = MinMaxScaler()
        Xtr[num_cols] = sc.fit_transform(Xtr[num_cols])
        Xte[num_cols] = sc.transform(Xte[num_cols])
    lr   = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(Xtr, y_tr)
    pred = lr.predict(Xte)
    results_sc[method] = {'Accuracy': round(accuracy_score(y_te, pred),4),
                          'F1-Score': round(f1_score(y_te, pred),4)}

print(f"{'Method':<18} {'Accuracy':>10} {'F1-Score':>10}")
print('-'*42)
for m, v in results_sc.items():
    note = ' ← DIPILIH' if m=='Tanpa Scaling' else ''
    print(f"{m:<18} {v['Accuracy']:>10} {v['F1-Score']:>10}{note}")


OHE cols: ['Type', 'Order Region', 'Customer State']
Dtypes sebelum OHE: Type              str
Order Region      str
Customer State    str
dtype: object
Method               Accuracy   F1-Score
------------------------------------------
Tanpa Scaling          0.6919     0.6766 ← DIPILIH
StandardScaler         0.6918     0.6767
MinMaxScaler           0.6918     0.6766


## Train Test Split

In [8]:
df = prepare_all(df)

Fitur baru ditambahkan: discount_per_unit
OHE cols: ['Type']
Dtypes sebelum OHE: Type    str
dtype: object
OHE cols: ['Type', 'Order Region', 'Customer State']
Dtypes sebelum OHE: Type              str
Order Region      str
Customer State    str
dtype: object
Skenario 1 setelah encoding: (180519, 9)
Skenario 2 setelah encoding: (180519, 78)

--- Skenario 1 (Pre-Shipment) ---
Train: 144,415 | Test: 36,104 | Fitur: 8

--- Skenario 2 (Full Order Profile) ---
Train: 144,415 | Test: 36,104 | Fitur: 77
